Script to compute automatic fluency features from audio transcriptions.

In [1]:
import pandas as pd
import os
import gzip
import glob
import tgt
import shutil
import re
import numpy as np

In [2]:
# Read textgrid and extract begin/end time of first three sentences
def read_tg_file_to_df(tg_file, file_encoding):

    # Read TextGrid file
    tg = tgt.io.read_textgrid(tg_file, encoding = file_encoding, include_empty_intervals=False)

    # Convert TextGrid file to Formatted Table (= df with on each row one interval)
    table = tgt.io.export_to_table(tg, separator='; ')
    formatted_table = [x.split('; ') for x in table.split('\n')]
    df = pd.DataFrame(formatted_table[1:], columns = formatted_table[0])

    return df

# Read textgrid and extract begin/end time of first three sentences
def read_tg_file_to_df_jasmin(tg_file):

    # Read TextGrid file
    try:
        tg = tgt.io.read_textgrid(tg_file, encoding='utf-8', include_empty_intervals=False)
    except:
        tg = tgt.io.read_textgrid(tg_file, encoding='utf-16', include_empty_intervals=False)

    # Convert TextGrid file to Formatted Table (= df with on each row one interval)
    table = tgt.io.export_to_table(tg, separator='; ')
    formatted_table = [x.split('; ') for x in table.split('\n')]

    return pd.DataFrame(formatted_table[1:], columns = formatted_table[0])

In [3]:
def get_measures_word_segmentation_cgn(tg_df_orth_trans, filename):
    # Preprocess intervals
    # 1) Remove intervals with only a _
    # 2) Remove intervals with only ggg, xxx or Xxx 
    # print(filename, tg_df_orth_trans[tg_df_orth_trans['text'].isin(['_', 'ggg', 'xxx', 'Xxx'])])
    tg_df_orth_trans = tg_df_orth_trans[~tg_df_orth_trans['text'].isin(['_', 'ggg', 'xxx', 'Xxx'])]

    # Create empty column called "filename"
    tg_df_orth_trans['file_name'] = [filename] * len(tg_df_orth_trans) 

    # Reset index
    tg_df_orth_trans = tg_df_orth_trans.reset_index(drop=True).drop(columns='tier_type', axis=1).rename({'tier_name': 'speaker_id'}, axis=1)

    startTimeFirstPrompt = tg_df_orth_trans.loc[0, 'start_time']
    endTimeLastPrompt = tg_df_orth_trans.loc[len(tg_df_orth_trans)-1, 'end_time']
    totalDuration = endTimeLastPrompt-startTimeFirstPrompt

    totalNrWords = len(tg_df_orth_trans)

    orthographic_transcription = " ". join(tg_df_orth_trans['text'])

    return totalNrWords, startTimeFirstPrompt, endTimeLastPrompt, totalDuration, orthographic_transcription

def get_measures_word_segmentation_jasmin(tg_df_orth_trans, filename, startTimeFirstPrompt, endTimeLastPrompt):
    # Select only orth trans of first story
    tg_df_orth_trans = tg_df_orth_trans[tg_df_orth_trans['start_time']>=startTimeFirstPrompt]
    tg_df_orth_trans = tg_df_orth_trans[tg_df_orth_trans['end_time']<=endTimeLastPrompt]

    # Preprocess intervals
    # 1) Remove intervals with only a _
    # 2) Remove intervals with only ggg, xxx or Xxx 
    # print(filename, tg_df_orth_trans[tg_df_orth_trans['text'].isin(['_', 'ggg', 'xxx', 'Xxx'])])
    tg_df_orth_trans = tg_df_orth_trans[~tg_df_orth_trans['text'].isin(['_', 'ggg', 'xxx', 'Xxx'])]

    # Create empty column called "filename"
    tg_df_orth_trans['file_name'] = [filename] * len(tg_df_orth_trans) 

    # Reset index
    tg_df_orth_trans = tg_df_orth_trans.reset_index(drop=True).drop(columns='tier_type', axis=1).rename({'tier_name': 'speaker_id'}, axis=1)

    totalNrWords = len(tg_df_orth_trans)

    orthographic_transcription = " ". join(tg_df_orth_trans['text'])

    return totalNrWords, orthographic_transcription, tg_df_orth_trans

def get_measures_phoneme_segmentation(tg_df_phon_trans, filename, startTimeFirstPrompt, endTimeLastPrompt):

    # Create empty column called "filename"
    tg_df_phon_trans['file_name'] = [filename] * len(tg_df_phon_trans) 

    # Remove non-story text at beginning (i.e., 'wil je nu ... voorlezen') from orthographic transcription
    tg_df_phon_trans = tg_df_phon_trans[tg_df_phon_trans['start_time']>=startTimeFirstPrompt]

    # Remove second text at the end
    tg_df_phon_trans = tg_df_phon_trans[tg_df_phon_trans['end_time']<=endTimeLastPrompt]

    # Reset index
    tg_df_phon_trans = tg_df_phon_trans.reset_index(drop=True).drop(columns='tier_type', axis=1).rename({'tier_name': 'speaker_id'}, axis=1)

    totalNrPhonemes = len(tg_df_phon_trans)

    return totalNrPhonemes

def detect_and_classify_silences(tg_df_orth_trans, filename):

    tg_df_orth_trans = tg_df_orth_trans.reset_index(drop=True)
    silenceMatrix = []

    for idx, row in tg_df_orth_trans.iterrows():
        if idx != 0:
            currentStartTime = row['start_time']

            previousEndTime = tg_df_orth_trans.loc[idx-1, 'end_time']

            if currentStartTime != previousEndTime:
                # There is a silence before the current word
                silence_start = previousEndTime
                silence_end = currentStartTime
                silence_dur = currentStartTime-previousEndTime

                # Explanations for silences
                text_before_silence = tg_df_orth_trans.loc[idx-1, 'text']
                sil_after_period = (tg_df_orth_trans.loc[idx-1, 'text'].find('.') != -1)
                sil_after_questionmark = (tg_df_orth_trans.loc[idx-1, 'text'].find('?') != -1)
                sil_after_exclamationmark = (tg_df_orth_trans.loc[idx-1, 'text'].find('!') != -1)

                # Add silences to seperate dataframe
                silenceMatrix.append([filename, silence_start, silence_end, silence_dur, text_before_silence, sil_after_period, sil_after_questionmark, sil_after_exclamationmark])

    return pd.DataFrame(silenceMatrix, columns = ["Filename", "silence_start", "silence_end", "silence_dur", "text_before_silence", "sil_after_period", "sil_after_questionmark", "sil_after_exclamationmark"])

# Define Non-silent pauses
fillers = ["ah", "aha", "ai", "au", "bah", "boe", "bwa", "eih", "eikes", "goh", "ha", "haha", "hé", "hè", 
            "hei", "ho", "hu", "hum", "hum hum", "jee", "o jee", "mm-hu", "mmm", "oeh", "oei", "oesje", "oh", 
            "o", "oho", "poeh", "pst", "sjt", "sst", "tut", "tut tut", "uh", "uhm", "uhu", "wauw", "woh", "zuh", "zulle"]

# Define incomprehensible and non-speech sounds
incomprehensible_and_non_speech_sounds = ["ggg", "Xxx", "xxx"]
# ggg = a non-speech sound produced by the speaker
# Xxx = one or more incomprehensible words or partial words
# xxx = an incomprehensible word that is clearly a title or proper name

def replace_asterisk_annotations(s):
    return re.sub('\*\w', '', s)

# Compute file-level scores from filled-pauses & silent-pauses
def getDuration(row):
    return row['end_time'] - row['start_time']

In [4]:
## Intersentential Pauses
# - These are expected pauses at the end of sentences. 
# - The follow when a period, exclamation mark or question mark is written in the prompt.
# - They are always silent pauses

def getIntersententialPauseMeasures(silentPausesDF, totalDuration):
    # nr/percentage of silent pauses at expected spot
    interSilentPausesDF = silentPausesDF[(silentPausesDF['sil_after_period'] == True) | (silentPausesDF['sil_after_questionmark'] == True) | (silentPausesDF['sil_after_exclamationmark'] == True)]
    nrExpSilentPauses = len(interSilentPausesDF)
    nrOfSentencesPerMinute = nrExpSilentPauses / (totalDuration/60)

    meanDurInterSilentPauses = round(interSilentPausesDF['silence_dur'].mean(),3)
    stdDurInterSilentPauses = round(interSilentPausesDF['silence_dur'].std(),3)

    # Pauses after period
    silentPausesAfterPeriodFileDF = silentPausesDF[(silentPausesDF['sil_after_period'] == True)]
    meanDurAfterPeriod = round(silentPausesAfterPeriodFileDF['silence_dur'].mean(),3)
    stdDurAfterPeriod = round(silentPausesAfterPeriodFileDF['silence_dur'].std(),3)

    # Pauses after exclamation mark
    silentPausesAfterPeriodFileDF = silentPausesDF[(silentPausesDF['sil_after_questionmark'] == True)]
    meanDurAfterExclMark = round(silentPausesAfterPeriodFileDF['silence_dur'].mean(),3)
    stdDurAfterExclMark = round(silentPausesAfterPeriodFileDF['silence_dur'].std(),3)

    # Pauses after question mark
    silentPausesAfterPeriodFileDF = silentPausesDF[(silentPausesDF['sil_after_exclamationmark'] == True)]
    meanDurAfterQuestionMark = round(silentPausesAfterPeriodFileDF['silence_dur'].mean(),3)
    stdDurAfterQuestionMark = round(silentPausesAfterPeriodFileDF['silence_dur'].std(),3)
    
    return {
                "nrOfSentencesPerMinute": nrOfSentencesPerMinute,
                "meanDurInterSilentPauses": meanDurInterSilentPauses,
                "stdDurInterSilentPauses": stdDurInterSilentPauses,
                "meanDurAfterPeriod" : meanDurAfterPeriod,
                "stdDurAfterPeriod" : stdDurAfterPeriod,
                "meanDurAfterExclMark" : meanDurAfterExclMark,
                "stdDurAfterExclMark": stdDurAfterExclMark,
                "meanDurAfterQuestionMark":meanDurAfterQuestionMark,
                "stdDurAfterQuestionMark": stdDurAfterQuestionMark
            }

def getIntrasententialPauseMeasures(silentPausesFileDF, totalDuration, totalNrWords, filledPausesFileDF):
    intraSilentPausesDF = silentPausesFileDF[(silentPausesFileDF['sil_after_period'] == False) | (silentPausesFileDF['sil_after_questionmark'] == False) | (silentPausesFileDF['sil_after_exclamationmark'] == False)]
    nrIntraSilentPauses = len(intraSilentPausesDF)

    nrIntraSilentPausesPerMin = nrIntraSilentPauses / (totalDuration/60)
    nrIntraSilentPausesPerWord = nrIntraSilentPauses / totalNrWords
    meanDurationIntraSilentPauses = round(intraSilentPausesDF['silence_dur'].mean(),3)
    stdDurationIntraSilentPauses = round(intraSilentPausesDF['silence_dur'].std(),3)

    # Filled Pauses    
    nrFilledPauses = len(filledPausesFileDF)

    filledPausesFileDF['dur'] = filledPausesFileDF.apply(lambda row: row['end_time']-row['start_time'], axis=1)

    if nrFilledPauses>0:
        nrIntraFilledPausesPerMin = nrFilledPauses / (totalDuration/60)
        meanDurationIntraFilledPauses = round(filledPausesFileDF['dur'].mean(),3)
        stdDurationIntraFilledPauses = round(filledPausesFileDF['dur'].std(),3)
    else:
        nrIntraFilledPausesPerMin = 0
        meanDurationIntraFilledPauses = np.nan
        stdDurationIntraFilledPauses = np.nan

    return {
                'nrIntraSilentPausesPerMin': nrIntraSilentPausesPerMin,
                'nrIntraSilentPausesPerWord': nrIntraSilentPausesPerWord,
                'meanDurationIntraSilentPauses': meanDurationIntraSilentPauses,
                'stdDurationIntraSilentPauses': stdDurationIntraSilentPauses,
                'nrIntraFilledPausesPerMin': nrIntraFilledPausesPerMin,
                'meanDurationIntraFilledPauses':meanDurationIntraFilledPauses,
                'stdDurationIntraFilledPauses': stdDurationIntraFilledPauses,
            }

In [5]:
def getArticulationRates(nrOfWords, nrOfSyllables, nrOfPhonemes, phonationTime):
    
    nrWordsPerMinute = nrOfWords / (phonationTime/60)
    nrWordsPerSecond = nrOfWords / phonationTime
    nrSyllPerMinute = nrOfSyllables / (phonationTime/60)
    nrSyllPerSecond = nrOfSyllables / phonationTime
    nrPhonemesPerMinute = nrOfPhonemes / (phonationTime/60)
    nrPhonemesPerSecond = nrOfPhonemes / phonationTime

    
    return {
                "ArtRate(nrWordsPerMinute)": nrWordsPerMinute,
                "ArtRate(nrSyllPerMinute)": nrSyllPerMinute,
                "ArtRate(nrPhonemesPerMinute)": nrPhonemesPerMinute,
            }    

def getSpeechRates(nrOfWords, nrOfSyllables, nrOfPhonemes, totalDuration):

    nrWordsPerMinute = nrOfWords / (totalDuration/60)
    nrWordsPerSecond = nrOfWords / totalDuration

    nrSyllPerMinute = nrOfSyllables / (totalDuration/60)
    nrSyllPerSecond = nrOfSyllables / totalDuration

    nrPhonemesPerMinute = nrOfPhonemes / (totalDuration/60)
    nrPhonemesPerSecond = nrOfPhonemes / totalDuration

    return {
                "SpeechRate(nrWordsPerMinute)": nrWordsPerMinute,
                "SpeechRate(nrSyllPerMinute)": nrSyllPerMinute,
                "SpeechRate(nrPhonemesPerMinute)": nrPhonemesPerMinute,
            } 

In [6]:
def getFileLevelPauseMeasures(filename, totalDuration, totalNrWords, filledPausesFileDF, silentPausesFileDF, nonSpeechIntervals):

    # Non-speech/incomprehensible intervals
    nrNonSpeechIntervals = len(nonSpeechIntervals)

    # if nrNonSpeechIntervals>0:
    #     durList = list(nonSpeechIntervals.apply(getDuration, axis=1))
    #     totalDurNonSpeechIntervals = round(sum(durList),3)
    #     meanDurNonSpeechIntervals = round(np.mean(durList),3)
    # else:
    #     totalDurNonSpeechIntervals = 0
    #     meanDurNonSpeechIntervals = 0
    
    # Silent Pauses
    # nrSilentPauses = len(silentPausesFileDF) 
    totalDurSilentPauses = round(sum(silentPausesFileDF['silence_dur']),3)
    # meanDurSilentPauses = round(silentPausesFileDF['silence_dur'].mean(),3)

    # # All pauses (Filled + Silent Pauses)
    # nrPauses = nrSilentPauses + nrFilledPauses
    # totalDurPauses = round(totalDurSilentPauses + totalDurFilledPauses,3)

    # Extract relevant measures
    intersententialMeasures = getIntersententialPauseMeasures(silentPausesFileDF, totalDuration)
    intrasententialMeasures = getIntrasententialPauseMeasures(silentPausesFileDF, totalDuration, totalNrWords, filledPausesFileDF)

    # Phonation time
    phonationTime = round(totalDuration-totalDurSilentPauses,3)

    return {
                'intersentential_pauses': intersententialMeasures,
                'intrasentential_pauses': intrasententialMeasures,
                'phonation_time': phonationTime,
            }
# Intersentential measures
# meanDurExpSilentPauses    Mean (in sec) of intersentential pause durations after period + excl. mark + question mark
# stdDurExpSilentPauses     Std (in sec) of intersentential pause durations after period + excl. mark + question mark

# meanDurAfterPeriod        Mean (in sec) of intersentential pause durations after period
# stdDurAfterPeriod         Std (in sec) of intersentential pause durations after period

# meanDurAfterExclMark      Mean (in sec) of intersentential pause durations after exclamation mark
# stdDurAfterExclMark       Std (in sec) of intersentential pause durations after exclamation mark

# meanDurAfterQuestionMark  Mean (in sec) of intersentential pause durations after question mark
# stdDurAfterQuestionMark   Std (in sec) of intersentential pause durations after question mark

# intrasentential measures

In [7]:
def readPitchFeatures(pitchDF, filename, allOrLastTag):
    
    # Select features of one file and convert them to dict
    filePitchDict = pitchDF.loc[filename].to_dict()
        
    # Remove irrelevant features from dict
    for key in ["dur_mean_"+allOrLastTag]:
        filePitchDict.pop(key)

    # filePitchDict['filename'] = filename

    return filePitchDict

# Main function

Inputs CGN - adult speech

In [25]:
# # Code to move .awd.gz files and unzip them

# origTgDir = '/vol/bigdata2/corpora2/CGN2/data/annot/text/awd/comp-o/nl'
# origTgFileList = glob.glob(os.path.join(origTgDir, '*.awd.gz'))

# for origTgFile in origTgFileList:
#     shutil.copy(origTgFile, os.path.join('/vol/tensusers5/wharmsen/DutchChildFluency/CGN2/awd_files_comp_o_nl', os.path.basename(origTgFile)))

# # gunzip *.gz

In [26]:
# Code to move .wav files

# origAudioDir = '/vol/bigdata2/corpora2/CGN2/data/audio/wav/comp-o/nl'
# origAudioFileList = glob.glob(os.path.join(origAudioDir, '*.wav'))

# for origAudioFile in origAudioFileList:
#     shutil.copy(origAudioFile, os.path.join('/vol/tensusers5/wharmsen/DutchChildFluency/CGN2/audio_files_comp_o_nl', os.path.basename(origAudioFile)))

# gunzip *.gz

In [ ]:
#Get list of all comp-o NL audio files
# audioDir = '/vol/bigdata2/corpora2/CGN2/data/audio/wav/comp-o/nl'
# audio_file_extension = '*.wav'

# # Get list of all comp-o NL basenames}
# audioFileList  = glob.glob(os.path.join(audioDir, audio_file_extension))
# fileBasenameList = [os.path.basename(audioFile).replace('.wav', '') for audioFile in audioFileList]
# print(len(fileBasenameList))

# Get list of all TextGrids with transcriptions of audio files
tgDir = './CGN2/awd_audio_comp_o_nl'
tg_file_extension = '*.awd'

# syllable DF
syllableDF = pd.read_csv(('./CGN2/fluency_scores/syllableDF.tsv'), sep='\t').set_index('filename')

# corpus
corpus = 'cgn'

# Output path
outputPath = 'automatic_measures.tsv'

Inputs JASMIN-CGN - child speech

In [ ]:
#Get list of all JASMIN NL audio files

# Get list of all TextGrids with transcriptions of audio files
tgDir = './JASMIN/'
tg_file_extension = '*.TextGrid'

# syllable DF
syllableDF = pd.read_csv(('orthographic_transcriptions_syllables.tsv'), sep='\t').set_index('filename')

# corpus
corpus = 'jasmin'

# output
outputPath = 'automatic_measures.tsv'


# JASMIN only: WCPM measures, AviLevel and Age
wcpmAgeAviDFAll = pd.read_csv('wcpm-measures.tsv', sep='\t').rename({'filename': 'fileName'}, axis=1).set_index('fileName')
wcpmAgeAviDFAll['aviLevelNumeric'] = wcpmAgeAviDFAll['aviLevel'].apply(lambda x: int(x.replace('AVI-', '')))


In [33]:
wcpmAgeAviDFAll.loc['fn000143', 'duration']

71.1579

# Start analysis

In [ ]:
tg_files = glob.glob(os.path.join(tgDir, tg_file_extension))
print(len(tg_files), 'to process')

71 to process


In [ ]:
filenames = []
pacePhrasingMeasuresAll = pd.DataFrame()

for tg_file in tg_files:

    filename = os.path.basename(tg_file).replace(tg_file_extension.replace('*', ''), '')
    filenames.append(filename)

    if corpus == 'cgn':
        tg_df = read_tg_file_to_df(tg_file, 'latin-1')
    elif corpus == 'jasmin':
        tg_df = read_tg_file_to_df_jasmin(tg_file)

    speaker_id = tg_df.loc[0,'tier_name']
    wordphon_tier = speaker_id + '_FON'
    phon_tier = speaker_id + '_SEG'

    if corpus == 'jasmin':

        """
        Only JASMIN - Tier 4: Prompt tier
        """
        tg_df_prompt = tg_df[tg_df['tier_name'] == 'Prompt'].astype({'start_time':float, 'end_time':float} ).reset_index().drop(columns='tier_type', axis=1).rename({'tier_name': 'speaker_id'}, axis=1)
        tg_df_prompt['file_name'] = [filename] * len(tg_df_prompt)

        # Get the start time of the first prompt
        startTimeFirstPrompt = tg_df_prompt.loc[0, 'start_time']
        endTimeLastPrompt = (tg_df_prompt.loc[len(tg_df_prompt)-1, 'end_time'])
        totalDuration = endTimeLastPrompt-startTimeFirstPrompt

        """
        Only JASMIN - Tier 1: Word segmentation of orthographic transcription => totalNrWords, totalDuration
        """ 
        # Preprocess
        tg_df_orth_trans = tg_df[tg_df['tier_name'] == speaker_id].astype({'start_time':float, 'end_time':float} )
        totalNrWords, orthographic_transcription, tg_df_orth_trans = get_measures_word_segmentation_jasmin(tg_df_orth_trans, filename, startTimeFirstPrompt, endTimeLastPrompt)

    elif corpus == 'cgn':

        """
        Only CGN - Tier 1: Word segmentation of orthographic transcription => totalNrWords, totalDuration
        """ 
        # Preprocess
        tg_df_orth_trans = tg_df[tg_df['tier_name'] == speaker_id].astype({'start_time':float, 'end_time':float} )
        totalNrWords, startTimeFirstPrompt, endTimeLastPrompt, totalDuration, orthographic_transcription = get_measures_word_segmentation_cgn(tg_df_orth_trans, filename)
    

    """
    Tier 3: Phoneme segmentation of phonetic transcription => totalNrPhonemes
    """  
    tg_df_phon_trans = tg_df[tg_df['tier_name'] == phon_tier].astype({'start_time':float, 'end_time':float} )
    totalNrPhonemes = get_measures_phoneme_segmentation(tg_df_phon_trans, filename, startTimeFirstPrompt, endTimeLastPrompt)

    """
    SyllableDF.csv: Read syllable count from orthographic transcriptions (computed using tool by Maarten van Gompel)
    """
    nrOfSyllables = syllableDF.loc[filename, 'totalNrSyllables']


    """
    Annotate pauses using context information
    """
    # Classify silent pauses as intersential or intrasentential
    silentPauseIntervals = detect_and_classify_silences(tg_df_orth_trans, filename)

    # Detect filled pauses
    filledPauseIntervals = tg_df_orth_trans[tg_df_orth_trans['text'].apply(replace_asterisk_annotations).isin(fillers)]

    # Detect non-speech sounds and incomprehensible speech
    nonSpeechIntervals = tg_df_orth_trans[tg_df_orth_trans['text'].apply(replace_asterisk_annotations).isin(incomprehensible_and_non_speech_sounds)]

    """
    Extract measures related to phrasing (pauses)
    """ 
    pauseMeasuresDict  = getFileLevelPauseMeasures(filename, totalDuration, totalNrWords, filledPauseIntervals, silentPauseIntervals, nonSpeechIntervals)
    
    phonationTime = pauseMeasuresDict['phonation_time']
    
    interSentDF = pd.DataFrame.from_dict(pauseMeasuresDict['intersentential_pauses'], orient='index', columns = [filename]).transpose()
    intraSentDF = pd.DataFrame.from_dict(pauseMeasuresDict['intrasentential_pauses'], orient='index', columns = [filename]).transpose()

    """
    Extract measures related to pace (speech rate and articulation rate)
    """
    speech_rates = getSpeechRates(totalNrWords, nrOfSyllables, totalNrPhonemes, totalDuration)
    #print(totalNrWords, nrOfSyllables, totalNrPhonemes, totalDuration)
    speechRateDF = pd.DataFrame.from_dict(speech_rates, orient='index', columns = [filename]).transpose()

    # print(totalNrWords, nrOfSyllables, totalNrPhonemes, phonationTime)
    articulation_rates = getArticulationRates(totalNrWords, nrOfSyllables, totalNrPhonemes, phonationTime)
    artRateDF = pd.DataFrame.from_dict(articulation_rates, orient='index', columns = [filename]).transpose()
    

    """
    Speaker information
    """
    fileInfoDF = pd.DataFrame.from_dict({"speakerID": speaker_id}, orient='index', columns = [filename]).transpose()

    if corpus == 'jasmin':
        #JASMIN only: WCPM, Age and Avi-Information
        wcpmAgeAviSeries = wcpmAgeAviDFAll.loc[filename, ['wcpm', 'perc_correct', 'aviLevel', 'age']]
        wcpmAgeAviDF = pd.DataFrame(wcpmAgeAviSeries).transpose()
        pacePhrasingMeasures = pd.concat([fileInfoDF, interSentDF, intraSentDF, speechRateDF, artRateDF, wcpmAgeAviDF], axis=1)
    elif corpus == 'cgn':
        pacePhrasingMeasures = pd.concat([fileInfoDF, interSentDF, intraSentDF, speechRateDF, artRateDF], axis=1)

    pacePhrasingMeasuresAll = pd.concat([pacePhrasingMeasuresAll, pacePhrasingMeasures], axis=0)

    print(totalNrWords, totalNrPhonemes, totalNrPhonemes/totalNrWords) #, interSentDF['nrOfSentencesPerMinute'].values()[0]*(totalDuration/60))

pacePhrasingMeasuresAll.to_csv(outputPath, sep='\t')
print('see output in ', outputPath)


/tmp/ipykernel_3943494/3181830341.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filledPausesFileDF['dur'] = filledPausesFileDF.apply(lambda row: row['end_time']-row['start_time'], axis=1)


204 695 3.406862745098039
252 913 3.623015873015873


/tmp/ipykernel_3943494/3181830341.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filledPausesFileDF['dur'] = filledPausesFileDF.apply(lambda row: row['end_time']-row['start_time'], axis=1)


206 686 3.3300970873786406
188 670 3.5638297872340425
129 422 3.2713178294573644
187 659 3.5240641711229945
218 751 3.444954128440367


/tmp/ipykernel_3943494/3181830341.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filledPausesFileDF['dur'] = filledPausesFileDF.apply(lambda row: row['end_time']-row['start_time'], axis=1)


...

157 498 3.171974522292994
191 637 3.3350785340314135


/tmp/ipykernel_3943494/3181830341.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filledPausesFileDF['dur'] = filledPausesFileDF.apply(lambda row: row['end_time']-row['start_time'], axis=1)


192 653 3.4010416666666665
237 794 3.350210970464135
194 681 3.5103092783505154
191 637 3.3350785340314135


/tmp/ipykernel_3943494/3181830341.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filledPausesFileDF['dur'] = filledPausesFileDF.apply(lambda row: row['end_time']-row['start_time'], axis=1)
/tmp/ipykernel_3943494/3181830341.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filledPausesFileDF['dur'] = filledPausesFileDF.apply(lambda row: row['end_time']-row['start_time'], axis=1)


202 658 3.257425742574257
177 605 3.4180790960451977
see output in  ./comp-q-read_nl_age7-11_nat/fluency-scores/story-level-final/final_pace_phrase_measures.tsv
